# Exploratory Data Analysis

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
from shapely import wkt
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import pandas as pd
import numpy as np
import folium
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("eda")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"
plot_path = '../plots'

### Full HVFHV Dataset:

In [ ]:
hvfhv_path = base_dir + '/developed/merged_data/full_hvfhv'
hvfhv_sdf = spark.read.parquet(hvfhv_path)
hvfhv_sdf.show(5)

### Demand Related Datasets:

Hourly demand dataset:

In [ ]:
hourly_demand_path = base_dir + '/developed/merged_data/hourly_pickup_demand'
hourly_demand = spark.read.parquet(hourly_demand_path)
hourly_demand_df = hourly_demand.toPandas()
hourly_demand_df.head()

Hourly demand among day of week dataset:

In [ ]:
hourly_demand_among_day_of_week_path = base_dir + '/developed/merged_data/hourly_demand_among_day_of_week'
hourly_demand_among_day_of_week = spark.read.parquet(hourly_demand_among_day_of_week_path)
hourly_demand_among_day_of_week_df = hourly_demand_among_day_of_week.toPandas()
hourly_demand_among_day_of_week_df.head()

Daily pickup demand by location ID dataset:

In [ ]:
daily_pickup_demand_by_location_path = base_dir + '/developed/merged_data/daily_pickup_demand_by_location.csv'
daily_pickup_demand_by_location_df = pd.read_csv(daily_pickup_demand_by_location_path)
daily_pickup_demand_by_location_df.head()

Daily dropoff demand by location ID dataset:

In [ ]:
daily_dropoff_demand_by_location_path = base_dir + '/developed/merged_data/daily_dropoff_demand_by_location.csv'
daily_dropoff_demand_by_location_df = pd.read_csv(daily_dropoff_demand_by_location_path)
daily_dropoff_demand_by_location_df.head()

Daily pickup demand by pickup date dataset:

In [ ]:
daily_demand_path = base_dir + '/developed/merged_data/daily_pickup_demand'
daily_demand = spark.read.parquet(daily_demand_path)
daily_demand.show(5)

Daily pickup demand by different building class dataset:

In [ ]:
demand_by_building_class_path = base_dir + '/developed/merged_data/daily_demand_by_building_class_df.csv'
demand_by_building_class_df = pd.read_csv(demand_by_building_class_path)
demand_by_building_class_df.head()

Monthly pickup demand dataset:

In [ ]:
month_path = base_dir + '/developed/merged_data/monthly_pickup_demand'
monthly_demand = spark.read.parquet(month_path)
monthly_demand_df = monthly_demand.toPandas()
monthly_demand_df.head()

### Revenue Related Datasets:

Hourly Revenue:

In [ ]:
hourly_revenue_path = base_dir + '/developed/merged_data/hourly_revenue'
hourly_revenue = spark.read.parquet(hourly_revenue_path)
hourly_revenue_df = hourly_revenue.toPandas()
hourly_revenue_df.head()

Hourly revenue among day of week dataset:

In [ ]:
hourly_revenue_among_day_of_week_path = base_dir + '/developed/merged_data/hourly_revenue_among_day_of_week'
hourly_revenue_among_day_of_week = spark.read.parquet(hourly_revenue_among_day_of_week_path)
hourly_revenue_among_day_of_week_df = hourly_revenue_among_day_of_week.toPandas()
hourly_revenue_among_day_of_week_df.head()

Daily revenue by location ID dataset:

In [ ]:
daily_revenue_by_location_df_path = base_dir + '/developed/merged_data/daily_revenue_by_location_df.csv'
daily_revenue_by_location_df = pd.read_csv(daily_revenue_by_location_df_path)
daily_revenue_by_location_df.head()

Daily revenue by pickup date dataset:

In [ ]:
daily_revenue_path = base_dir + '/developed/merged_data/daily_revenue'
daily_revenue = spark.read.parquet(daily_revenue_path)
daily_revenue.show(5)

Monthly revenue dataset:

In [ ]:
month_path = base_dir + '/developed/merged_data/monthly_revenue'
monthly_revenue = spark.read.parquet(month_path)
monthly_revenue_df = monthly_revenue.toPandas()
monthly_revenue_df.head()

### Utilization Rate Dataset:

In [ ]:
utilization_rate_path = base_dir + '/developed/merged_data/utilization_rate'
utilization_rate = spark.read.parquet(utilization_rate_path)
utilization_rate_df = utilization_rate.toPandas()
utilization_rate_df.head()

### Geographic Zone Dataset:

In [ ]:
zone_gdf_path = base_dir + '/developed/merged_data/zone_gdf.csv'
zone_gdf = pd.read_csv(zone_gdf_path)
zone_gdf['geometry'] = zone_gdf['geometry'].apply(wkt.loads)
zone_gdf.head()

### PLUTO Related Datasets:

In [ ]:
pluto_df_path = base_dir + '/developed/merged_data/pluto_df.csv'
pluto_df = pd.read_csv(pluto_df_path)
pluto_df.head()

### Weather Related Datasets:

In [ ]:
weather_path = base_dir + '/developed/merged_data/hourly_demand_by_weather'
hourly_demand_by_weather = spark.read.parquet(weather_path)
hourly_demand_by_weather_df = hourly_demand_by_weather.toPandas()
hourly_demand_by_weather_df.head()

# Plots Related to Demand:

### Compare the Peak Hours of Pickup Demand for Weekday and Weekend:

In [ ]:
# Group by pickup_hour and day_type to calculate average hourly demand for Weekend and Weekday
avg_demand = hourly_demand_df.groupby(['pickup_hour', 'day_type'])['mean_hourly_demand']\
                             .mean().reset_index()

# Pivot the table so we have separate columns for Weekend and Weekday
avg_demand_pivot = avg_demand.pivot(index='pickup_hour', columns='day_type', 
                                    values='mean_hourly_demand')

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(avg_demand_pivot.index, avg_demand_pivot['Weekend'], 
         marker='o', label='Weekend', color='blue')
plt.plot(avg_demand_pivot.index, avg_demand_pivot['Weekday'], 
         marker='o', label='Weekday', color='red')

# Add labels and title
plt.xlabel('Hour')
plt.ylabel('Average Hourly Demand')
plt.title('Average Hourly Demand by Day Type', weight='bold', fontsize=14)
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True)

# Save and display the plot
file_name = 'hourly_demand_by_day_type.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

### Compare Hourly Pickup Demand Among Day of Week:

In [ ]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Sort the DataFrame by day_of_week and pickup_hour
hourly_demand_among_day_of_week_df.sort_values(by=['pickup_hour'], inplace=True)

# Create the plot
plt.figure(figsize=(7, 4))

# Define colors for each day of the week
colors = {
    'Monday': 'purple',
    'Tuesday': 'orange',
    'Wednesday': 'green',
    'Thursday': 'red',
    'Friday': 'blue',
    'Saturday': 'black',
    'Sunday': 'pink'
}

# Plot each day of the week
for day in days:
    day_data = hourly_demand_among_day_of_week_df[hourly_demand_among_day_of_week_df['day_of_week'] == day]
    sns.lineplot(x='pickup_hour', y='day_by_hour_demand', data=day_data, label=day, color=colors[day], errorbar=None)

plt.xlabel('Hour', fontsize=14)
plt.ylabel('Demand', fontsize=14)
plt.title('Hourly Demand by Day of the Week', weight='bold', fontsize=14)
plt.legend(title='Day of the Week')
plt.grid(True)

# Adjust the size of the tick labels on both axes
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Save and display the plot
plt.tight_layout()

file_name = 'hourly_demand_among_day_of_week.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

### Compare Daily Pickup Demand by Location ID:

In [ ]:
zone_gdf = gpd.GeoDataFrame(zone_gdf, geometry='geometry')
zone_gdf = zone_gdf.drop_duplicates('location_id')
geoJSON = zone_gdf[['location_id', 'geometry']].drop_duplicates('location_id').to_json()

In [ ]:
# Create map with CartoDB Positron tiles (light color background)
m = folium.Map(location=[40.73, -73.74], 
               tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
               attr="Map tiles by CartoDB, under CC BY 3.0. Data by OpenStreetMap, under ODbL.",
               zoom_start=10,
               zoom_control=False,
               width='100%',
               height='500px')

# Add title
title = '''
        <h3 align="left" style="font-size:16px"><b>
        Daily Pickup Demand of High Volume for Hire Vehicle in NYC</b></h3>'''
m.get_root().html.add_child(folium.Element(title))

# Create Choropleth layer
c = folium.Choropleth(
    geo_data=geoJSON,  # geoJSON 
    name='choropleth',  # plot name
    data=daily_pickup_demand_by_location_df,  # data source
    columns=['location_id', 'daily_demand'],  # required columns
    key_on='properties.location_id',  # geoJSON property key
    bins=9,
    fill_color='RdPu',  # color scheme
    line_opacity=0.3,
    nan_fill_color='Lightgreen',
    legend_name='Daily Pickup Demand of High Volume for Hire Vehicle in NYC'
)

# Add layer to map
c.add_to(m)

In [ ]:
daily_pickup_demand_by_location_df.head()

This shows the locations for highest pickup daily demand are in location 138, 132, 79, 61, and 230. We will label them in the map.

In [ ]:
# Filter the GeoDataFrame for specific location_ids
selected_locations = [138, 132, 79, 61, 230]
filtered_zone_gdf = zone_gdf.loc[zone_gdf['location_id'].isin(selected_locations)].copy()

# Calculate centroids if not already calculated
filtered_zone_gdf.loc[:, 'centroid'] = filtered_zone_gdf['geometry'].apply(lambda x: (x.centroid.y, x.centroid.x))

# Add markers to the map for the selected locations
for _, row in filtered_zone_gdf.iterrows():
    location_id = row['location_id']
    zone_name = row['zone']
    coord = row['centroid']
    folium.Marker(location=coord, popup=f"{zone_name} (ID: {location_id})").add_to(m)

# Save and display the map
file_name = 'daily_pickup_demand_by_location_id.html'
file_path = os.path.join(plot_path, file_name)
m.save(file_path)

m

From the above plot, we could know that the daily demand varies by different boroughs. So we will look at the daily demand by boroughs later.

In [ ]:
filtered_zone_gdf

By the previous analysis, we know the building class with highest daily demand is B, A, C, S, and D. Next we will look at the building classes for each `zone` (the zone with highest pickup demand) in the previous dataframe:

In [ ]:
highest_daily_demand_location_zones = ['LaGuardia Airport', 
                                       'Crown Heights North', 
                                       'East Village', 
                                       'JFK Airport', 
                                       'Times Sq/Theatre District']
highest_daily_demand_building_class = ['B', 'A', 'C', 'S', 'D']

for sub_class in highest_daily_demand_building_class:
    sub_df = pluto_df[pluto_df['building_class'] == sub_class]
    zone_borough = sub_df[['zone', 'borough']].drop_duplicates()

    zones = zone_borough['zone'].unique()
    boroughs = zone_borough['borough'].unique()

    print('Building Class', sub_class+':')
    print('Boroughs:', boroughs)
    print('Zones:')
    for zone in highest_daily_demand_location_zones:
        if (zone in zones) == True:
            print(zone, 'yes')
    print('\n')

### Compare Daily Dropoff Demand by Location ID:

In [ ]:
# Create map with CartoDB Positron tiles (light color background)
m = folium.Map(location=[40.73, -73.74], 
               tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
               attr="Map tiles by CartoDB, under CC BY 3.0. Data by OpenStreetMap, under ODbL.",
               zoom_start=10,
               zoom_control=False,
               width='100%',
               height='500px')

# Add title
title = '''
        <h3 align="left" style="font-size:16px"><b>
        Daily Dropoff Demand of High Volume for Hire Vehicle in NYC</b></h3>'''
m.get_root().html.add_child(folium.Element(title))

# Create Choropleth layer
c = folium.Choropleth(
    geo_data=geoJSON,  # geoJSON 
    name='choropleth',  # plot name
    data=daily_dropoff_demand_by_location_df,  # data source
    columns=['location_id', 'daily_demand'],  # required columns
    key_on='properties.location_id',  # geoJSON property key
    bins=9,
    fill_color='RdPu',  # color scheme
    line_opacity=0.3,
    nan_fill_color='Lightgreen',
    legend_name='Daily Dropoff Demand of High Volume for Hire Vehicle in NYC'
)

# Add layer to map
c.add_to(m)

In [ ]:
daily_dropoff_demand_by_location_df.head()

This shows the locations for highest dropoff daily demand are in location 132, 138, 61, 68, and 79. We will label them in the map.

In [ ]:
# Filter the GeoDataFrame for specific location_ids
selected_locations = [132, 138, 61, 68, 79]
filtered_zone_gdf = zone_gdf.loc[zone_gdf['location_id'].isin(selected_locations)].copy()

# Calculate centroids if not already calculated
filtered_zone_gdf.loc[:, 'centroid'] = filtered_zone_gdf['geometry'].apply(lambda x: (x.centroid.y, x.centroid.x))

# Add markers to the map for the selected locations
for _, row in filtered_zone_gdf.iterrows():
    location_id = row['location_id']
    zone_name = row['zone']
    coord = row['centroid']
    folium.Marker(location=coord, popup=f"{zone_name} (ID: {location_id})").add_to(m)

# Save and display the map
file_name = 'daily_dropoff_demand_by_location_id.html'
file_path = os.path.join(plot_path, file_name)
m.save(file_path)

m

In [ ]:
filtered_zone_gdf

By the previous analysis, we know the building class with highest daily demand is B, A, C, S, and D. Next we will look at the building classes for each `zone` (the zone with highest dropoff demand) in the previous dataframe:

In [ ]:
highest_daily_demand_location_zones = ['LaGuardia Airport', 
                                       'Crown Heights North', 
                                       'East Village', 
                                       'JFK Airport', 
                                       'East Chelsea']
highest_daily_demand_building_class = ['B', 'A', 'C', 'S', 'D']

for sub_class in highest_daily_demand_building_class:
    sub_df = pluto_df[pluto_df['building_class'] == sub_class]
    zone_borough = sub_df[['zone', 'borough']].drop_duplicates()

    zones = zone_borough['zone'].unique()
    boroughs = zone_borough['borough'].unique()

    print('Building Class', sub_class+':')
    print('Boroughs:', boroughs)
    print('Zones:')
    for zone in highest_daily_demand_location_zones:
        if (zone in zones) == True:
            print(zone, 'yes')
    print('\n')

### Compare Daily Demand by Building Class:

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(demand_by_building_class_df['building_class'], demand_by_building_class_df['daily_demand'], color='skyblue')

# Add labels to each bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, f'{yval:.0e}', ha='center', va='bottom', rotation=45)

plt.title('Daily Demand by Building Class', weight='bold', fontsize=14)
plt.xlabel('Building Class')
plt.ylabel('Daily Demand')

# Save and display the plot
file_name = 'daily_demand_by_building_class.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Apply log transform to daily demand
demand_by_building_class_df['log_daily_demand'] = np.log1p(demand_by_building_class_df['daily_demand'])

# Plot
plt.figure(figsize=(10, 6))
bars = plt.bar(demand_by_building_class_df['building_class'], demand_by_building_class_df['log_daily_demand'], color='skyblue')

# Add the logarithmic value to each column
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, f'{yval:.2f}', ha='center', va='bottom', rotation=30)

plt.title('Log of Daily Demand by Building Class', weight='bold', fontsize=14)
plt.xlabel('Building Class')
plt.ylabel('Log of Daily Demand')

# Save and display the plot
file_name = 'log_daily_demand_by_building_class.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

### Daily Pickup Demand by Different Boroughs:

#### Plots:

In [ ]:
# Get all location IDs for each borough
borough_id = {}
for borough in zone_gdf['borough'].unique():
    borough_id[borough] = zone_gdf[zone_gdf['borough'] == borough]['location_id'].unique().tolist()

# Placeholder for how your DataFrame is structured
daily_demand_by_borough = {}

# Create a sample dataframe for each borough
for borough, ids in borough_id.items():
    daily_demand_by_borough[borough] = daily_demand.filter(daily_demand['PULocationID'].isin(ids)).toPandas()

# Initialize the plot
idx = 0
fig, ax = plt.subplots(len(borough_id.keys()), figsize=(14, 24))

# Define holiday colors
holiday_colors = {
    'Independence Day': 'orange',
    'Labor Day': 'green',
    'Election Day': 'gray',
    'Thanksgiving Day': 'brown',
    'Christmas Day': 'red'
}

# Define holidays
holidays = {
    'Independence Day': datetime.datetime(2023, 7, 4),
    'Labor Day': datetime.datetime(2023, 9, 4),
    'Election Day': datetime.datetime(2023, 11, 7),
    'Thanksgiving Day': datetime.datetime(2023, 11, 24),
    'Christmas Day': datetime.datetime(2023, 12, 25)
}

# Find top 3 days with highest daily demand for each borough
top_days_by_borough = {}

for borough, df in daily_demand_by_borough.items():
    # Sum up the pickup counts grouped by date
    df = df.groupby('pickup_date').agg({'daily_demand': 'sum'}).reset_index()
    
    # Find the top 3 days with highest daily demand
    top_days = df.sort_values(by='daily_demand', ascending=False).head(3)
    top_days_by_borough[borough] = top_days

    # Create line plot
    sns.lineplot(x='pickup_date', y='daily_demand', data=df, ax=ax[idx])

    # Highlight specific public holidays with vertical lines
    for holiday_name, holiday_date in holidays.items():
        ax[idx].axvline(holiday_date, color=holiday_colors[holiday_name], linestyle='--', label=f'{holiday_name}: {holiday_date.date()}')

    # Add title for each subplot
    ax[idx].set_title(f'Daily Pickup Demand in {borough}', weight='bold', fontsize=14)
    
    # Adjust legend
    ax[idx].legend(loc='upper left', bbox_to_anchor=(1, 1))  # Position legend outside the plot area

    idx += 1

# Save and display the plot
plt.tight_layout(rect=[0, 0, 0.85, 1]) 

file_name = 'daily_demand_by_borough.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

#### Find top 3 days with highest daily demand for each borough:

In [ ]:
for borough, top_days in top_days_by_borough.items():
    print(f"Top 3 days with highest daily demand in {borough}:")
    print(top_days)
    print("\n")

### Daily Pickup Demand:

#### Plot:

In [ ]:
# Define holiday colors
holiday_colors = {
    'Before Halloween': 'green',
    'Thanksgiving Day': 'orange',
    'Christmas Day': 'red'
}

# Define holidays
holidays = {
    'Before Halloween': datetime.datetime(2023, 10, 28),
    'Thanksgiving Day': datetime.datetime(2023, 11, 24),
    'Christmas Day': datetime.datetime(2023, 12, 25)
}

# Combine data for all boroughs into a single DataFrame
combined_df = pd.concat(daily_demand_by_borough.values())

# Sum up the pickup counts grouped by date across all boroughs
combined_df = combined_df.groupby('pickup_date').agg({'daily_demand': 'sum'}).reset_index()

# Initialize the plot
plt.figure(figsize=(8, 4))

# Create line plot
sns.lineplot(x='pickup_date', y='daily_demand', data=combined_df)

# Highlight specific public holidays with vertical lines
for holiday_name, holiday_date in holidays.items():
    plt.axvline(holiday_date, color=holiday_colors[holiday_name], linestyle='--', label=f'{holiday_name}: {holiday_date.date()}')

# Add title and adjust legend
plt.title('Daily Pickup Demand', weight='bold', fontsize=15)
plt.legend()  # Position legend outside the plot area 

# Adjust the size of the axis labels
plt.xlabel('Pickup Date', fontsize=13)
plt.ylabel('Daily Demand', fontsize=13)

# Adjust the size of the tick labels on both axes
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Save and display the plot
file_name = 'daily_demand.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

#### Find top 3 days with highest daily demand:

In [ ]:
# Find the top 3 days with the highest daily demand
top_3_days = combined_df.sort_values(by='daily_demand', ascending=False).head(3)

# Output the result
print("Top 3 days with the highest daily demand:")
print(top_3_days)

### Compare Monthly Demand:

In [ ]:
# Create the plot
plt.figure(figsize=(8, 4))
bars = plt.bar(monthly_demand_df['month'], monthly_demand_df['monthly_demand'], color='skyblue')

# Annotate each bar with the corresponding demand value
for bar in bars:
    yval = bar.get_height()
    formatted_yval = f"{int(yval):,}"  # Format the number with commas
    plt.text(bar.get_x() + bar.get_width()/2, yval, formatted_yval, va='bottom', ha='center')

plt.xlabel('Month', fontsize=10)
plt.ylabel('Demand', fontsize=10)
plt.title('Monthly Demand', weight='bold', fontsize=14)
plt.xticks()

# Save and display the plot
plt.tight_layout()

file_name = 'monthly_pickup_demand.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

# Plots Related to Revenue:

### Compare the Peak of Revenue for Weekday and Weekend:

In [ ]:
# Group by pickup_hour and day_type to calculate average hourly demand for Weekend and Weekday
avg_revenue = hourly_revenue_df.groupby(['pickup_hour', 'day_type'])['mean_hourly_revenue']\
                             .mean().reset_index()

# Pivot the table so we have separate columns for Weekend and Weekday
avg_revenue_pivot = avg_revenue.pivot(index='pickup_hour', columns='day_type', 
                                      values='mean_hourly_revenue')

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(avg_revenue_pivot.index, avg_revenue_pivot['Weekend'], 
         marker='o', label='Weekend', color='blue')
plt.plot(avg_revenue_pivot.index, avg_revenue_pivot['Weekday'], 
         marker='o', label='Weekday', color='red')

# Add labels and title
plt.xlabel('Hour')
plt.ylabel('Average Hourly Revenue')
plt.title('Average Hourly Revenue by Day Type', weight='bold', fontsize=14)
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True)

# Save and display the plot
file_name = 'hourly_revenue_by_day_type.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

### Compare Hourly Revenue Among Day of Week:

In [ ]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Sort the DataFrame by day_of_week and pickup_hour
hourly_revenue_among_day_of_week_df.sort_values(by=['pickup_hour'], inplace=True)

# Create the plot
plt.figure(figsize=(7, 4))

# Define colors for each day of the week
colors = {
    'Monday': 'purple',
    'Tuesday': 'orange',
    'Wednesday': 'green',
    'Thursday': 'red',
    'Friday': 'blue',
    'Saturday': 'black',
    'Sunday': 'pink'
}

# Plot each day of the week
for day in days:
    day_data = hourly_revenue_among_day_of_week_df[hourly_revenue_among_day_of_week_df['day_of_week'] == day]
    sns.lineplot(x='pickup_hour', y='day_by_hour_revenue', data=day_data, label=day, color=colors[day], errorbar=None)

plt.xlabel('Hour', fontsize=14)
plt.ylabel('Revenue ($)', fontsize=14)
plt.title('Hourly Revenue by Day of the Week', weight='bold', fontsize=14)
plt.legend(title='Day of the Week')
plt.grid(True)

# Adjust the size of the tick labels on both axes
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Save and display the plot
plt.tight_layout()

file_name = 'hourly_revenue_among_day_of_week.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

### Compare Daily Revenue by Location ID:

In [ ]:
# Create map with CartoDB Positron tiles (light color background)
m = folium.Map(location=[40.73, -73.74], 
               tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
               attr="Map tiles by CartoDB, under CC BY 3.0. Data by OpenStreetMap, under ODbL.",
               zoom_start=10,
               zoom_control=False,
               width='100%',
               height='500px')

# Add title
title = '''
        <h3 align="left" style="font-size:16px"><b>
        Daily Revenue of High Volume for Hire Vehicle in NYC</b></h3>'''
m.get_root().html.add_child(folium.Element(title))

# Create Choropleth layer
c = folium.Choropleth(
    geo_data=geoJSON,  # geoJSON 
    name='choropleth',  # plot name
    data=daily_revenue_by_location_df,  # data source
    columns=['location_id', 'daily_revenue'],  # required columns
    key_on='properties.location_id',  # geoJSON property key
    bins=9,
    fill_color='RdPu',  # color scheme
    line_opacity=0.3,
    nan_fill_color='Lightgreen',
    legend_name='Daily Revenue of High Volume for Hire Vehicle in NYC'
)

# Add layer to map
c.add_to(m)

In [ ]:
daily_revenue_by_location_df.head()

This shows the locations for highest daily revenue are in location 132, 138, 230, 161, and 79. We will label them in the map.

In [ ]:
# Filter the GeoDataFrame for specific location_ids
selected_locations = [132, 138, 230, 161, 79]
filtered_zone_gdf = zone_gdf.loc[zone_gdf['location_id'].isin(selected_locations)].copy()

# Calculate centroids if not already calculated
filtered_zone_gdf.loc[:, 'centroid'] = filtered_zone_gdf['geometry'].apply(lambda x: (x.centroid.y, x.centroid.x))

# Add markers to the map for the selected locations
for _, row in filtered_zone_gdf.iterrows():
    location_id = row['location_id']
    zone_name = row['zone']
    coord = row['centroid']
    folium.Marker(location=coord, popup=f"{zone_name} (ID: {location_id})").add_to(m)

# Save and display the map
file_name = 'daily_revenue_by_location_id.html'
file_path = os.path.join(plot_path, file_name)
m.save(file_path)

m

From the above plot, we could know that the daily revenue varies by different boroughs. So we will look at the daily revenue by boroughs later.

In [ ]:
filtered_zone_gdf

Thus, the futher analysis of revenue will focus on Manhattan and Queens.

By the previous analysis, we know the building class with highest daily revenue is B, A, C, S, and D. Next we will look at the building classes for each `zone` (the zone with highest daily revenue) in the previous dataframe:

In [ ]:
highest_daily_revenue_location_zones = ['LaGuardia Airport', 
                                       'Midtown Center', 
                                       'East Village', 
                                       'JFK Airport', 
                                       'Times Sq/Theatre District']
highest_daily_revenue_location_class = ['B', 'A', 'C', 'S', 'D']

for sub_class in highest_daily_revenue_location_class:
    sub_df = pluto_df[pluto_df['building_class'] == sub_class]
    zone_borough = sub_df[['zone', 'borough']].drop_duplicates()

    zones = zone_borough['zone'].unique()
    boroughs = zone_borough['borough'].unique()

    print('Building Class', sub_class+':')
    print('Boroughs:', boroughs)
    print('Zones:')
    for zone in highest_daily_revenue_location_zones:
        if (zone in zones) == True:
            print(zone, 'yes')
    print('\n')

### Daily Revenue by Different Boroughs:

#### Plots:

In [ ]:
# Get all location IDs for each borough
borough_id = {}
for borough in zone_gdf['borough'].unique():
    borough_id[borough] = zone_gdf[zone_gdf['borough'] == borough]['location_id'].unique().tolist()

# Placeholder for how your DataFrame is structured
daily_revenue_by_borough = {}

# Create a sample dataframe for each borough
for borough, ids in borough_id.items():
    daily_revenue_by_borough[borough] = daily_revenue.filter(daily_revenue['PULocationID'].isin(ids)).toPandas()

# Initialize the plot
idx = 0
fig, ax = plt.subplots(len(borough_id.keys()), figsize=(14, 24))

# Find top 3 days with highest daily revenue for each borough
top_days_by_borough = {}

for borough, df in daily_revenue_by_borough.items():
    # Sum up the pickup counts grouped by date
    df = df.groupby('pickup_date').agg({'daily_revenue': 'sum'}).reset_index()
    
    # Find the top 3 days with highest daily revenue
    top_days = df.sort_values(by='daily_revenue', ascending=False).head(3)
    top_days_by_borough[borough] = top_days

    # Create line plot
    sns.lineplot(x='pickup_date', y='daily_revenue', data=df, ax=ax[idx])

    # Highlight specific public holidays with vertical lines
    for holiday_name, holiday_date in holidays.items():
        ax[idx].axvline(holiday_date, color=holiday_colors[holiday_name], linestyle='--', label=f'{holiday_name}: {holiday_date.date()}')

    # Add title for each subplot
    ax[idx].set_title(f'Daily Revenue in {borough}', weight='bold', fontsize=14)
    
    # Adjust legend
    ax[idx].legend(loc='upper left', bbox_to_anchor=(1, 1))  # Position legend outside the plot area

    idx += 1

# Save and display the plot
plt.tight_layout(rect=[0, 0, 0.85, 1])

file_name = 'daily_revenue_by_borough.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

#### Find top 3 days with highest daily revenue for each borough:

In [ ]:
for borough, top_days in top_days_by_borough.items():
    print(f"Top 3 days with highest daily revenue in {borough}:")
    print(top_days)
    print("\n")

### Daily Revenue:

#### Plot:

In [ ]:
# Define holiday colors
holiday_colors = {
    'Travis Scott': 'green',
    'Thanksgiving Day': 'orange',
    'Christmas Day': 'red'
}

# Define holidays
holidays = {
    'Travis Scott': datetime.datetime(2023, 9, 29),
    'Thanksgiving Day': datetime.datetime(2023, 11, 24),
    'Christmas Day': datetime.datetime(2023, 12, 25)
}

# Combine data for all boroughs into a single DataFrame
combined_df = pd.concat(daily_revenue_by_borough.values())

# Sum up the pickup counts grouped by date across all boroughs
combined_df = combined_df.groupby('pickup_date').agg({'daily_revenue': 'sum'}).reset_index()

# Initialize the plot
plt.figure(figsize=(8, 4))

# Create line plot
sns.lineplot(x='pickup_date', y='daily_revenue', data=combined_df)

# Highlight specific public holidays with vertical lines
for holiday_name, holiday_date in holidays.items():
    plt.axvline(holiday_date, color=holiday_colors[holiday_name], linestyle='--', label=f'{holiday_name}: {holiday_date.date()}')

# Add title and adjust legend
plt.title('Daily Revenue', weight='bold', fontsize=15)
plt.legend()  # Position legend outside the plot area 
              # loc='upper left', bbox_to_anchor=(1, 1)

# Adjust the size of the axis labels
plt.xlabel('Pickup Date', fontsize=13)
plt.ylabel('Daily Revenue ($)', fontsize=13)

# Adjust the size of the tick labels on both axes
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Save and display the plot
file_name = 'daily_revenue.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

#### Find top 3 days with highest daily revenue:

In [ ]:
# Find the top 3 days with the highest daily revenue
top_3_days = combined_df.sort_values(by='daily_revenue', ascending=False).head(3)

# Output the result
print("Top 3 days with the highest daily revenue:")
print(top_3_days)

### Compare Monthly Revenue:

In [ ]:
# Create the plot
plt.figure(figsize=(8, 4))
bars = plt.bar(monthly_revenue_df['month'], monthly_revenue_df['monthly_revenue'], color='pink')

# Annotate each bar with the corresponding revenue value
for bar in bars:
    yval = bar.get_height()
    formatted_yval = f"{int(yval):,}"  # Format the number with commas
    plt.text(bar.get_x() + bar.get_width()/2, yval, formatted_yval, va='bottom', ha='center')

plt.xlabel('Month', fontsize=10)
plt.ylabel('Revenue', fontsize=10)
plt.title('Monthly Revenue', weight='bold', fontsize=14)
plt.xticks()

# Save and display the plot
plt.tight_layout()

file_name = 'monthly_revenue.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

# Plots Related Utilization Rate:

### Compare Market Share by Different Companies:

In [ ]:
# Count the occurrences of each hvfhs_license_num
hvfhs_counts = hvfhv_sdf.groupBy('hvfhs_license_num').count().toPandas()

# Mapping the hvfhs_license_num to company names
hvfhs_counts['company'] = hvfhs_counts['hvfhs_license_num'].map({1: 'Lyft', 0: 'Uber'})

# Plotting the pie chart
plt.figure(figsize=(4, 3))
plt.pie(hvfhs_counts['count'], labels=hvfhs_counts['company'], 
        autopct='%1.1f%%', startangle=140, colors=['#ff7f0e', '#1f77b4'],
        textprops={'fontsize': 14})  # Adjust the fontsize here
plt.title('Market Shares for Lyft and Uber', weight='bold', fontsize=14)

# Save and display the plot
file_name = 'market_share.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

### Compare Utilization Rate by Different Companies:

In [ ]:
# Mapping the hvfhs_license_num to company names
utilization_rate_df['company'] = utilization_rate_df['hvfhs_license_num'].map({1: 'Lyft', 0: 'Uber'})

# Plotting the utilization rate over time for each company
plt.figure(figsize=(5, 3))
for company in utilization_rate_df['company'].unique():
    subset = utilization_rate_df[utilization_rate_df['company'] == company]
    plt.plot(subset['pickup_hour'], subset['avg_utilization_rate'], label=company)

# Formatting the plot
plt.title('Hourly Utilization Rates by Lyft and Uber', weight='bold', fontsize=15)
plt.xlabel('Hour', fontsize=13)
plt.ylabel('Utilization Rate', fontsize=13)
plt.legend(title='Company', labels=['Uber', 'Lyft'])  # 0 -> Uber, 1 -> Lyft
plt.tight_layout()
plt.grid(True)

# Adjust the size of the tick labels on both axes
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

# Save and display the plot
file_name = 'utilization_rate.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

# Plots Related to Weather:

In [ ]:
sorted_df = hourly_demand_by_weather_df.sort_values(by='avg_hourly_demand')

# Create the plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x='AvgHourlyTemp', y='avg_hourly_demand', data=sorted_df)
plt.xlabel('Average Hourly Temperature (°C)')
plt.ylabel('Average Hourly Demand')
plt.title('Averaged Hourly Demand vs Temperature', weight='bold', fontsize=14)
plt.grid(True)

# Save and display the plot
file_name = 'hourly_demand_by_temperature.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
sorted_df = hourly_demand_by_weather_df.sort_values(by='AvgHourlyPrecipitation')

# Create the plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x='AvgHourlyPrecipitation', y='avg_hourly_demand', data=sorted_df)

plt.xlabel('Average Hourly Precipitation (inches)')
plt.ylabel('Average Hourly Demand')
plt.title('Averaged Hourly Demand vs Precipitation', weight='bold', fontsize=14)
plt.grid(True)

# Rotate x-axis tick labels
plt.xticks(rotation=90)  # Rotate x-axis labels by 45 degrees

# Save and display the plot
file_name = 'hourly_demand_by_precipitation.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
sorted_df = hourly_demand_by_weather_df.sort_values(by='AvgHourlyVisibility')

# Create the plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x='AvgHourlyVisibility', y='avg_hourly_demand', data=hourly_demand_by_weather_df)
plt.xlabel('Average Hourly Visibility (miles)')
plt.ylabel('Average Hourly Demand')
plt.title('Averaged Hourly Demand vs Visibility', weight='bold', fontsize=14)
plt.grid(True)

# Save and display the plot
file_name = 'hourly_demand_by_visibility.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
sorted_df = hourly_demand_by_weather_df.sort_values(by='AvgHourlyWindSpeed')

# Create the plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x='AvgHourlyWindSpeed', y='avg_hourly_demand', data=hourly_demand_by_weather_df)
plt.xlabel('Average Hourly Wind Speed (mph)')
plt.ylabel('Average Hourly Demand')
plt.title('Averaged Hourly Demand vs Wind Speed', weight='bold', fontsize=14)
plt.grid(True)

# Save and display the plot
file_name = 'hourly_demand_by_wind_speed.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()